# Preprocessing y diseno de alertas

Objetivo: estudiar `Datasets.xlsx` y dejar una base reproducible para una app standalone que se actualiza cada dia y entrega cada semana un calendario organizado de alertas explicables.

La logica separa dos mundos de producto:
- **Commodities**: recurrencia de compra, anomalias, promiscuidad de compra, compras marginales frente al potencial.
- **Productos tecnicos**: riesgo de perdida, cambios de frecuencia o volumen, desapariciones y actividad anomala.

Cada alerta debe traer: semana/dia sugerido, urgencia, cliente, contacto disponible, bloque/familia/categoria/producto cuando aplique, razon explicable y metricas que la sustentan.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("Datasets.xlsx")
assert DATA_PATH.exists(), f"No existe {DATA_PATH.resolve()}"

## 1. Inventario del Excel

Primero leemos todas las pestanas y revisamos forma, columnas, tipos y nulos. Esto tambien detecta pestanas que son tablas dinamicas o agregados auxiliares.

In [ ]:
xl = pd.ExcelFile(DATA_PATH)
raw = {sheet: pd.read_excel(DATA_PATH, sheet_name=sheet) for sheet in xl.sheet_names}

summary = []
for name, df in raw.items():
    summary.append({
        "sheet": name,
        "rows": len(df),
        "cols": df.shape[1],
        "columns": list(df.columns),
        "null_cells": int(df.isna().sum().sum()),
    })

pd.DataFrame(summary)

In [ ]:
for name in xl.sheet_names:
    print(f"\n--- {name} ---")
    display(raw[name].head())

## 2. Limpieza y normalizacion

Notas iniciales:
- `Clientes` trae la primera fila como cabecera real; `Column2` se interpreta como zona/codigo comercial.
- `Ventas (cliente)` y `Ventas (producto)` parecen tablas dinamicas. Las conservamos como referencia, pero el modelo de alertas se construye desde `Ventas`.
- El contacto de cliente no aparece en estas pestanas; dejamos una columna `contacto_cliente` vacia para que la app pueda rellenarla si se conecta a CRM/ERP.

In [ ]:
def get_sheet(name_or_prefix):
    exact = [name for name in raw if name.lower() == name_or_prefix.lower()]
    if exact:
        return raw[exact[0]].copy()
    matches = [name for name in raw if name.lower().startswith(name_or_prefix.lower())]
    if not matches:
        raise KeyError(f"No hay pestana que empiece por {name_or_prefix!r}")
    return raw[matches[0]].copy()

clientes_raw = get_sheet("Clientes")
clientes = clientes_raw.iloc[1:].copy()
clientes.columns = ["id_cliente", "zona", "provincia"]
clientes["id_cliente"] = pd.to_numeric(clientes["id_cliente"], errors="coerce").astype("Int64")
clientes["zona"] = pd.to_numeric(clientes["zona"], errors="coerce").astype("Int64")
clientes["provincia"] = clientes["provincia"].astype("string").str.strip()
clientes["contacto_cliente"] = pd.NA
clientes = clientes.dropna(subset=["id_cliente"]).drop_duplicates("id_cliente")

productos = get_sheet("Productos")
productos.columns = ["id_producto", "bloque", "categoria", "familia"]
productos["id_producto"] = pd.to_numeric(productos["id_producto"], errors="coerce").astype("Int64")
for col in ["bloque", "categoria", "familia"]:
    productos[col] = productos[col].astype("string").str.strip()
productos = productos.dropna(subset=["id_producto"]).drop_duplicates("id_producto")

ventas = get_sheet("Ventas")
ventas.columns = ["num_fact", "fecha", "id_cliente", "id_producto", "unidades", "valor"]
ventas["fecha"] = pd.to_datetime(ventas["fecha"], errors="coerce")
ventas["id_cliente"] = pd.to_numeric(ventas["id_cliente"], errors="coerce").astype("Int64")
ventas["id_producto"] = pd.to_numeric(ventas["id_producto"], errors="coerce").astype("Int64")
ventas["unidades"] = pd.to_numeric(ventas["unidades"], errors="coerce")
ventas["valor"] = pd.to_numeric(ventas["valor"], errors="coerce")
ventas["year"] = ventas["fecha"].dt.year
ventas["year_month"] = ventas["fecha"].dt.to_period("M").astype("string")
ventas["es_devolucion"] = (ventas["unidades"] < 0) | (ventas["valor"] < 0)
ventas["es_linea_cero"] = (ventas["unidades"] == 0) | (ventas["valor"] == 0)

potencial = get_sheet("Potencial")
potencial.columns = ["id_cliente", "familia_potencial", "categoria_potencial", "potencial_anual"]
potencial["id_cliente"] = pd.to_numeric(potencial["id_cliente"], errors="coerce").astype("Int64")
potencial["potencial_anual"] = pd.to_numeric(potencial["potencial_anual"], errors="coerce").fillna(0)

campanias = get_sheet("Campa")
campanias.columns = ["campania", "fecha_inicio", "fecha_fin"]
campanias["fecha_inicio"] = pd.to_datetime(campanias["fecha_inicio"], errors="coerce")
campanias["fecha_fin"] = pd.to_datetime(campanias["fecha_fin"], errors="coerce")

ventas_enriquecidas = (
    ventas
    .merge(productos, on="id_producto", how="left", validate="many_to_one")
    .merge(clientes, on="id_cliente", how="left", validate="many_to_one")
)

ventas_enriquecidas["campania"] = pd.NA
for row in campanias.itertuples(index=False):
    mask = ventas_enriquecidas["fecha"].between(row.fecha_inicio, row.fecha_fin)
    ventas_enriquecidas.loc[mask, "campania"] = ventas_enriquecidas.loc[mask, "campania"].fillna(row.campania)

potencial_cliente = (
    potencial.groupby("id_cliente", as_index=False)
    .agg(
        potencial_anual_total=("potencial_anual", "sum"),
        potencial_categorias=("categoria_potencial", "nunique"),
        potencial_familias=("familia_potencial", "nunique"),
    )
)

ventas_enriquecidas = ventas_enriquecidas.merge(potencial_cliente, on="id_cliente", how="left", validate="many_to_one")
ventas_enriquecidas["potencial_anual_total"] = ventas_enriquecidas["potencial_anual_total"].fillna(0)

ventas_enriquecidas.head()

## 3. Checks de calidad

Estos checks separan problemas de datos de senales de negocio. Por ejemplo, valores negativos pueden ser devoluciones/abonos y tambien pueden alimentar alertas.

In [ ]:
quality = {
    "fecha_min": ventas_enriquecidas["fecha"].min(),
    "fecha_max": ventas_enriquecidas["fecha"].max(),
    "lineas_ventas": len(ventas_enriquecidas),
    "clientes_unicos": ventas_enriquecidas["id_cliente"].nunique(),
    "productos_unicos": ventas_enriquecidas["id_producto"].nunique(),
    "facturas_unicas": ventas_enriquecidas["num_fact"].nunique(),
    "productos_sin_master": ventas_enriquecidas["bloque"].isna().sum(),
    "clientes_sin_master": ventas_enriquecidas["provincia"].isna().sum(),
    "lineas_devolucion": int(ventas_enriquecidas["es_devolucion"].sum()),
    "lineas_cero": int(ventas_enriquecidas["es_linea_cero"].sum()),
}
pd.Series(quality)

In [ ]:
ventas_enriquecidas.groupby(["year", "bloque"], dropna=False).agg(
    lineas=("num_fact", "size"),
    clientes=("id_cliente", "nunique"),
    productos=("id_producto", "nunique"),
    unidades=("unidades", "sum"),
    valor=("valor", "sum"),
).reset_index().sort_values(["year", "bloque"])

## 4. Tablas base para comportamiento cliente-producto

Trabajamos con ventas positivas para modelar recurrencia y consumo. Las devoluciones se guardan en otra tabla porque son una senal propia.

In [ ]:
ventas_pos = ventas_enriquecidas[(ventas_enriquecidas["unidades"] > 0) & (ventas_enriquecidas["valor"] > 0)].copy()
ventas_dev = ventas_enriquecidas[ventas_enriquecidas["es_devolucion"]].copy()

grain = ["id_cliente", "bloque", "categoria", "familia"]

ordenes = (
    ventas_pos.groupby(grain + ["num_fact"], dropna=False, as_index=False)
    .agg(
        fecha=("fecha", "min"),
        valor=("valor", "sum"),
        unidades=("unidades", "sum"),
        productos_distintos=("id_producto", "nunique"),
    )
)

def gap_stats(group):
    dates = pd.Series(group["fecha"].dropna().sort_values().unique())
    if len(dates) < 2:
        return pd.Series({"median_gap_days": np.nan, "mean_gap_days": np.nan, "last_gap_days": np.nan})
    gaps = dates.diff().dt.days.dropna()
    return pd.Series({
        "median_gap_days": float(gaps.median()),
        "mean_gap_days": float(gaps.mean()),
        "last_gap_days": float(gaps.iloc[-1]),
    })

base_cf = (
    ordenes.groupby(grain, dropna=False)
    .agg(
        primera_compra=("fecha", "min"),
        ultima_compra=("fecha", "max"),
        pedidos=("num_fact", "nunique"),
        valor_total=("valor", "sum"),
        unidades_total=("unidades", "sum"),
        ticket_medio=("valor", "mean"),
    )
    .reset_index()
)

gaps = ordenes.groupby(grain, dropna=False).apply(gap_stats, include_groups=False).reset_index()
base_cf = base_cf.merge(gaps, on=grain, how="left")
base_cf = base_cf.merge(clientes, on="id_cliente", how="left")
base_cf = base_cf.merge(potencial_cliente, on="id_cliente", how="left")
base_cf["potencial_anual_total"] = base_cf["potencial_anual_total"].fillna(0)

base_cf["dias_desde_ultima"] = (ventas_enriquecidas["fecha"].max() - base_cf["ultima_compra"]).dt.days
base_cf["fecha_esperada"] = base_cf["ultima_compra"] + pd.to_timedelta(base_cf["median_gap_days"].round(), unit="D")
base_cf.sort_values("valor_total", ascending=False).head(20)

## 5. Reglas explicables de alertas

Estas reglas son deliberadamente transparentes. En una segunda fase se pueden combinar con modelos probabilisticos, pero la primera version ya permite entregar un calendario accionable y auditable.

Urgencia propuesta:
- **Critica**: senal fuerte de perdida/anomalia o retraso importante en cliente recurrente.
- **Alta**: accion recomendada esta semana.
- **Media**: seguimiento preventivo.
- **Baja**: oportunidad o higiene comercial.

In [ ]:
def urgency_from_score(score):
    if score >= 90:
        return "Critica"
    if score >= 70:
        return "Alta"
    if score >= 45:
        return "Media"
    return "Baja"

def alert_record(row, alert_type, score, reason, suggested_date, metrics=None):
    metrics = metrics or {}
    return {
        "fecha_sugerida": pd.Timestamp(suggested_date).normalize(),
        "semana": pd.Timestamp(suggested_date).to_period("W-MON").start_time,
        "urgencia": urgency_from_score(score),
        "score": int(round(score)),
        "tipo_alerta": alert_type,
        "id_cliente": row.get("id_cliente"),
        "contacto_cliente": row.get("contacto_cliente", pd.NA),
        "zona": row.get("zona", pd.NA),
        "provincia": row.get("provincia", pd.NA),
        "bloque": row.get("bloque", pd.NA),
        "categoria": row.get("categoria", pd.NA),
        "familia": row.get("familia", pd.NA),
        "razon": reason,
        "metricas": metrics,
    }

def make_alerts(run_date=None, horizon_days=7):
    if run_date is None:
        # En produccion seria pd.Timestamp.today().normalize(). Aqui usamos el dia siguiente al maximo historico.
        run_date = ventas_enriquecidas["fecha"].max().normalize() + pd.Timedelta(days=1)
    run_date = pd.Timestamp(run_date).normalize()
    horizon_end = run_date + pd.Timedelta(days=horizon_days - 1)
    alerts = []

    # 1) Commodities: compras recurrentes que deberian ocurrir pronto o ya van tarde.
    recurrent = base_cf[
        (base_cf["bloque"] == "Commodities")
        & (base_cf["pedidos"] >= 4)
        & (base_cf["median_gap_days"].between(7, 180, inclusive="both"))
        & base_cf["fecha_esperada"].notna()
    ].copy()
    recurrent["dias_retraso"] = (run_date - recurrent["fecha_esperada"]).dt.days
    due = recurrent[(recurrent["fecha_esperada"] <= horizon_end) & (recurrent["dias_retraso"] >= -7)]
    for _, row in due.iterrows():
        lateness = max(row["dias_retraso"], 0)
        score = min(95, 55 + lateness * 2 + min(row["pedidos"], 15))
        reason = (
            f"Cliente recurrente en commodity: {int(row['pedidos'])} pedidos, "
            f"mediana entre compras {row['median_gap_days']:.0f} dias, "
            f"ultima compra {row['ultima_compra'].date()}, compra esperada {row['fecha_esperada'].date()}."
        )
        alerts.append(alert_record(row, "Commodity - compra recurrente esperada", score, reason, max(run_date, row["fecha_esperada"]), {
            "pedidos": int(row["pedidos"]),
            "median_gap_days": round(row["median_gap_days"], 1),
            "dias_retraso": int(lateness),
            "valor_total": round(row["valor_total"], 2),
        }))

    # 2) Commodities: promiscuidad/switching dentro de categoria.
    recent_start = run_date - pd.Timedelta(days=180)
    recent_comm = ventas_pos[(ventas_pos["bloque"] == "Commodities") & (ventas_pos["fecha"] >= recent_start)]
    prod_mix = recent_comm.groupby(["id_cliente", "categoria"], as_index=False).agg(
        productos_distintos=("id_producto", "nunique"),
        valor_180d=("valor", "sum"),
        pedidos_180d=("num_fact", "nunique"),
    )
    top_share = (
        recent_comm.groupby(["id_cliente", "categoria", "id_producto"], as_index=False)["valor"].sum()
        .sort_values("valor", ascending=False)
        .groupby(["id_cliente", "categoria"], as_index=False)
        .first()
        .rename(columns={"valor": "valor_top_producto", "id_producto": "producto_top"})
    )
    prod_mix = prod_mix.merge(top_share, on=["id_cliente", "categoria"], how="left")
    prod_mix["share_top"] = prod_mix["valor_top_producto"] / prod_mix["valor_180d"]
    prod_mix = prod_mix.merge(clientes, on="id_cliente", how="left")
    promiscuos = prod_mix[(prod_mix["productos_distintos"] >= 3) & (prod_mix["share_top"] < 0.65) & (prod_mix["valor_180d"] > 0)]
    for _, row in promiscuos.sort_values("valor_180d", ascending=False).head(50).iterrows():
        score = 45 + min(row["productos_distintos"] * 6, 25) + min(row["valor_180d"] / 5000, 20)
        row = row.copy()
        row["bloque"] = "Commodities"
        reason = (
            f"Compra repartida en {int(row['productos_distintos'])} productos de {row['categoria']} en 180 dias; "
            f"el producto principal solo concentra {row['share_top']:.0%} del valor."
        )
        alerts.append(alert_record(row, "Commodity - cliente promiscuo", score, reason, run_date + pd.Timedelta(days=1), {
            "productos_distintos": int(row["productos_distintos"]),
            "share_top": round(row["share_top"], 3),
            "valor_180d": round(row["valor_180d"], 2),
        }))

    # 3) Commodities: compra marginal frente a potencial anual.
    annual_start = run_date - pd.Timedelta(days=365)
    actual_365 = ventas_pos[ventas_pos["fecha"] >= annual_start].groupby("id_cliente", as_index=False).agg(valor_365d=("valor", "sum"))
    marginal = potencial_cliente.merge(actual_365, on="id_cliente", how="left").merge(clientes, on="id_cliente", how="left")
    marginal["valor_365d"] = marginal["valor_365d"].fillna(0)
    marginal["captura_potencial"] = marginal["valor_365d"] / marginal["potencial_anual_total"].replace(0, np.nan)
    threshold_pot = marginal["potencial_anual_total"].quantile(0.75)
    marginal = marginal[(marginal["potencial_anual_total"] >= threshold_pot) & (marginal["captura_potencial"] < 0.20)]
    for _, row in marginal.sort_values("potencial_anual_total", ascending=False).head(75).iterrows():
        row = row.copy()
        row["bloque"] = "Commodities"
        score = 35 + min(row["potencial_anual_total"] / 10000, 35) + (20 if row["valor_365d"] == 0 else 0)
        reason = (
            f"Potencial anual alto ({row['potencial_anual_total']:,.0f}) pero compra ultimos 365 dias "
            f"de {row['valor_365d']:,.0f}; captura estimada {row['captura_potencial']:.0%}."
        )
        alerts.append(alert_record(row, "Commodity - compra marginal vs potencial", score, reason, run_date + pd.Timedelta(days=2), {
            "potencial_anual_total": round(row["potencial_anual_total"], 2),
            "valor_365d": round(row["valor_365d"], 2),
            "captura_potencial": None if pd.isna(row["captura_potencial"]) else round(row["captura_potencial"], 3),
        }))

    # 4) Productos tecnicos: riesgo de perdida/desaparicion por retraso frente a frecuencia historica.
    tech = base_cf[
        (base_cf["bloque"] == "Productos Tecnicos")
        | (base_cf["bloque"] == "Productos Técnicos")
    ].copy()
    tech = tech[(tech["pedidos"] >= 3) & tech["median_gap_days"].notna()]
    tech["dias_retraso"] = (run_date - tech["fecha_esperada"]).dt.days
    tech_loss = tech[tech["dias_retraso"] > np.maximum(30, tech["median_gap_days"] * 0.75)]
    for _, row in tech_loss.sort_values("dias_retraso", ascending=False).head(100).iterrows():
        score = min(98, 60 + min(row["dias_retraso"] / 2, 30) + min(row["valor_total"] / 50000, 8))
        reason = (
            f"Riesgo de perdida tecnica: {int(row['pedidos'])} pedidos historicos, "
            f"mediana {row['median_gap_days']:.0f} dias, ultima compra {row['ultima_compra'].date()}, "
            f"esperada {row['fecha_esperada'].date()}, retraso {row['dias_retraso']:.0f} dias."
        )
        alerts.append(alert_record(row, "Tecnico - riesgo de perdida", score, reason, run_date, {
            "pedidos": int(row["pedidos"]),
            "median_gap_days": round(row["median_gap_days"], 1),
            "dias_retraso": int(row["dias_retraso"]),
            "valor_total": round(row["valor_total"], 2),
        }))

    # 5) Productos tecnicos: caida de volumen/valor ultimos 90 dias vs 90 dias previos.
    tech_sales = ventas_pos[(ventas_pos["bloque"] == "Productos Técnicos") | (ventas_pos["bloque"] == "Productos Tecnicos")].copy()
    current_start = run_date - pd.Timedelta(days=90)
    previous_start = run_date - pd.Timedelta(days=180)
    current = tech_sales[tech_sales["fecha"].between(current_start, run_date - pd.Timedelta(days=1))].groupby(["id_cliente", "categoria", "familia"], as_index=False).agg(valor_actual_90d=("valor", "sum"), pedidos_actual_90d=("num_fact", "nunique"))
    previous = tech_sales[tech_sales["fecha"].between(previous_start, current_start - pd.Timedelta(days=1))].groupby(["id_cliente", "categoria", "familia"], as_index=False).agg(valor_previo_90d=("valor", "sum"), pedidos_previo_90d=("num_fact", "nunique"))
    drops = previous.merge(current, on=["id_cliente", "categoria", "familia"], how="left").fillna({"valor_actual_90d": 0, "pedidos_actual_90d": 0})
    drops["caida_valor_pct"] = 1 - (drops["valor_actual_90d"] / drops["valor_previo_90d"].replace(0, np.nan))
    drops = drops.merge(clientes, on="id_cliente", how="left")
    drops = drops[(drops["valor_previo_90d"] > drops["valor_previo_90d"].quantile(0.60)) & (drops["caida_valor_pct"] >= 0.50)]
    for _, row in drops.sort_values("caida_valor_pct", ascending=False).head(100).iterrows():
        row = row.copy()
        row["bloque"] = "Productos Técnicos"
        score = 55 + min(row["caida_valor_pct"] * 35, 35) + (8 if row["pedidos_actual_90d"] == 0 else 0)
        reason = (
            f"Caida tecnica de valor: 90 dias recientes {row['valor_actual_90d']:,.0f} vs "
            f"90 dias previos {row['valor_previo_90d']:,.0f}; descenso {row['caida_valor_pct']:.0%}."
        )
        alerts.append(alert_record(row, "Tecnico - caida de volumen/valor", score, reason, run_date + pd.Timedelta(days=1), {
            "valor_actual_90d": round(row["valor_actual_90d"], 2),
            "valor_previo_90d": round(row["valor_previo_90d"], 2),
            "caida_valor_pct": round(row["caida_valor_pct"], 3),
            "pedidos_actual_90d": int(row["pedidos_actual_90d"]),
            "pedidos_previo_90d": int(row["pedidos_previo_90d"]),
        }))

    # 6) Todas las familias: devoluciones o abonos recientes anormalmente altos.
    dev_recent = ventas_dev[ventas_dev["fecha"] >= run_date - pd.Timedelta(days=30)]
    dev = dev_recent.groupby(["id_cliente", "bloque", "categoria", "familia"], dropna=False, as_index=False).agg(
        devoluciones_30d=("num_fact", "nunique"),
        valor_devolucion_30d=("valor", "sum"),
        unidades_devolucion_30d=("unidades", "sum"),
    )
    if len(dev):
        dev["abs_valor_devolucion_30d"] = dev["valor_devolucion_30d"].abs()
        dev = dev.merge(clientes, on="id_cliente", how="left")
        threshold_dev = dev["abs_valor_devolucion_30d"].quantile(0.75)
        dev = dev[(dev["devoluciones_30d"] >= 2) | (dev["abs_valor_devolucion_30d"] >= threshold_dev)]
        for _, row in dev.sort_values("abs_valor_devolucion_30d", ascending=False).head(100).iterrows():
            score = 50 + min(row["devoluciones_30d"] * 8, 24) + min(row["abs_valor_devolucion_30d"] / 3000, 20)
            reason = (
                f"Actividad anomala reciente: {int(row['devoluciones_30d'])} facturas con devolucion/abono "
                f"en 30 dias por {row['valor_devolucion_30d']:,.0f}."
            )
            alerts.append(alert_record(row, "Anomalia - devoluciones recientes", score, reason, run_date, {
                "devoluciones_30d": int(row["devoluciones_30d"]),
                "valor_devolucion_30d": round(row["valor_devolucion_30d"], 2),
                "unidades_devolucion_30d": round(row["unidades_devolucion_30d"], 2),
            }))

    alerts_df = pd.DataFrame(alerts)
    if alerts_df.empty:
        return alerts_df
    priority = {"Critica": 0, "Alta": 1, "Media": 2, "Baja": 3}
    alerts_df["prioridad"] = alerts_df["urgencia"].map(priority)
    alerts_df = alerts_df.sort_values(["fecha_sugerida", "prioridad", "score"], ascending=[True, True, False]).drop(columns="prioridad")
    return alerts_df.reset_index(drop=True)

alertas = make_alerts()
alertas.head(25)

## 6. Calendario semanal de alertas

La app puede ejecutar `make_alerts(run_date=fecha_de_hoy)` cada dia y, una vez por semana, renderizar esta vista agrupada por dia y urgencia.

In [ ]:
def weekly_calendar(alerts_df, week_start=None, max_items_per_day=30):
    if alerts_df.empty:
        return alerts_df
    if week_start is None:
        week_start = alerts_df["fecha_sugerida"].min().normalize()
    week_start = pd.Timestamp(week_start).normalize()
    week_end = week_start + pd.Timedelta(days=6)
    out = alerts_df[alerts_df["fecha_sugerida"].between(week_start, week_end)].copy()
    out["dia_semana"] = out["fecha_sugerida"].dt.day_name()
    out["rank_dia"] = out.groupby("fecha_sugerida")["score"].rank(method="first", ascending=False)
    out = out[out["rank_dia"] <= max_items_per_day]
    cols = [
        "fecha_sugerida", "dia_semana", "urgencia", "score", "tipo_alerta", "id_cliente",
        "contacto_cliente", "zona", "provincia", "bloque", "categoria", "familia", "razon", "metricas"
    ]
    return out[cols].sort_values(["fecha_sugerida", "urgencia", "score"], ascending=[True, True, False]).reset_index(drop=True)

calendario_semana = weekly_calendar(alertas)
calendario_semana.head(100)

In [ ]:
if not alertas.empty:
    display(alertas.groupby(["tipo_alerta", "urgencia"]).size().rename("n_alertas").reset_index().sort_values(["tipo_alerta", "urgencia"]))
    display(calendario_semana.groupby(["fecha_sugerida", "urgencia"]).size().rename("n_alertas").reset_index())

## 7. Artefactos para la app

Guardamos parquet/csv con tablas limpias y el calendario. En produccion conviene persistir:
- ventas enriquecidas: base historica trazable.
- base cliente-familia: features de comportamiento.
- alertas: cola diaria completa.
- calendario semanal: entrega curada para negocio.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ventas_enriquecidas.to_csv(OUTPUT_DIR / "ventas_enriquecidas.csv", index=False)
base_cf.to_csv(OUTPUT_DIR / "features_cliente_familia.csv", index=False)
alertas.to_csv(OUTPUT_DIR / "alertas_diarias.csv", index=False)
calendario_semana.to_csv(OUTPUT_DIR / "calendario_semanal_alertas.csv", index=False)

list(OUTPUT_DIR.glob("*.csv"))

## 8. Siguientes decisiones de negocio

Preguntas que hay que cerrar con la empresa antes de convertir esto en app:
1. Definicion exacta de contacto de cliente: no viene en el Excel actual.
2. Ventana operativa del calendario: semana natural, semana comercial o proximos 7 dias desde la ejecucion.
3. Politica de saturacion: maximo de alertas por comercial/zona/dia.
4. Umbrales por familia: los umbrales globales son una primera aproximacion; conviene calibrarlos con historico validado por ventas.
5. Tratamiento de campanas: separar compra organica vs compra inducida para no confundir recurrencia real.